In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore")

# Display all input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Introduction

This notebook presents the initial baseline solution for the Smart MCQ Solver Challenge, developed as part of the Deep Learning & Generative AI Project. The competition focuses on building intelligent machine learning models capable of predicting the top three most probable answers for challenging multiple-choice questions.

Each question consists of a prompt and five answer options (A, B, C, D, and E). Instead of predicting only the single correct answer, the model must rank the three most likely answers. Performance is evaluated using Mean Average Precision at 3 (MAP@3), which rewards models that rank the correct answer higher in the prediction list.

# Objectives

The objective of this project is to develop an intelligent multiple-choice question answering system capable of accurately predicting the top three most probable answers for each question using Natural Language Processing (NLP), Machine Learning, Deep Learning, and Generative AI techniques. The project involves understanding and preprocessing textual data, performing exploratory data analysis, building and evaluating baseline, classical machine learning, neural network, and transformer-based models, and comparing their performance using the Mean Average Precision at 3 (MAP@3) evaluation metric. Throughout the project, emphasis is placed on developing a systematic machine learning pipeline, improving model performance through experimentation and optimization, and gaining a comprehensive understanding of modern AI methodologies and their practical applications in automated question answering.

# Exploratory Data Analysis (EDA)

In [2]:
# Import Required Libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train Shape :", train.shape)
print("Test Shape :", test.shape)
print("Sample Submission Shape :", sample.shape)

Train Shape : (2000, 8)
Test Shape : (500, 7)
Sample Submission Shape : (500, 2)


In [ ]:
train.head()

**Dataset Information**

In [ ]:
train.info()

he training dataset consists of 2,000 multiple-choice questions with 8 columns. Each row represents one question and contains a unique question identifier (id), the question prompt (prompt), five answer options (A, B, C, D, and E), and the corresponding correct answer (answer). The id column is of integer type (int64), while the remaining seven columns are of object type (object) since they contain textual data. The dataset structure confirms that it is well-organized

**Missing Values**

In [ ]:
missing = train.isnull().sum().to_frame("Missing Values")
missing

The missing value analysis indicates that none of the columns contain missing or null values. Every column has 2,000 non-null entries, confirming that the dataset is complete. Since there are no missing values, no additional data imputation or preprocessing is required to handle incomplete records. This ensures that all observations can be used during model training and evaluation without any loss of information.

**Duplicate Rows**

In [ ]:
print("Duplicate Rows :", train.duplicated().sum())

The duplicate row analysis shows that the dataset contains zero duplicate records. This indicates that every question in the dataset is unique, preventing redundant information from influencing the learning process. The absence of duplicate entries contributes to better model generalization and reduces the risk of biased training caused by repeated samples.

**Class Distribution**

In [ ]:
answer_counts = train["answer"].value_counts()

print(answer_counts)

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(answer_counts.index, answer_counts.values)

plt.title("Distribution of Correct Answers")
plt.xlabel("Answer Option")
plt.ylabel("Count")

plt.show()

The distribution of the correct answer labels reveals that the dataset is reasonably balanced, although the classes are not perfectly equal. Option B is the most frequent correct answer with 490 occurrences, followed by C (459), A (369), D (358), and E (324). While the class frequencies vary slightly, the imbalance is not severe enough to significantly affect model training.

**Prompt Length Analysis**

In [ ]:
train["Prompt Length"] = train["prompt"].str.len()

train["Prompt Length"].describe()

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(train["Prompt Length"], bins=30)

plt.title("Distribution of Prompt Length")
plt.xlabel("Characters")
plt.ylabel("Frequency")

plt.show()

The prompt length analysis measures the number of characters present in each question prompt. The dataset contains 2,000 prompts, with an average length of approximately 118 characters. The shortest prompt contains 19 characters, while the longest consists of 337 characters, indicating considerable variation in question complexity and detail. Half of the prompts contain 111 characters or fewer, while 75% of the prompts have a length of 141 characters or less.

**Prompt Word Count Analysis**

In [ ]:
train["Prompt Word Count"] = train["prompt"].str.split().apply(len)

train["Prompt Word Count"].describe()

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(train["Prompt Word Count"], bins=30)

plt.title("Prompt Word Count Distribution")
plt.xlabel("Words")
plt.ylabel("Frequency")

plt.show()

The prompt word count analysis examines the number of words contained in each question prompt. On average, each prompt consists of approximately 18 words, with the shortest prompt containing only 3 words and the longest containing 51 words. The median prompt length is 17 words, indicating that most questions are relatively concise while still providing sufficient contextual information.

**Option Length Analysis**

In [ ]:
for option in ["A","B","C","D","E"]:
    train[f"{option}_Length"] = train[option].str.split().apply(len)

train[[f"{c}_Length" for c in ["A","B","C","D","E"]]].describe()

**Average option lengths**

In [ ]:
avg_lengths = []

for option in ["A","B","C","D","E"]:
    avg_lengths.append(train[f"{option}_Length"].mean())

plt.figure(figsize=(6,4))

plt.bar(["A","B","C","D","E"], avg_lengths)

plt.title("Average Word Length of Answer Options")

plt.xlabel("Option")

plt.ylabel("Average Number of Words")

plt.show()

The answer option length analysis evaluates the number of words present in each of the five answer choices (A–E). The results show that all five options have a similar average length of approximately 26 words, indicating that the dataset has been designed with fairly balanced answer choices.

# Dummy Submission

In [ ]:
import pandas as pd

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

sample["Prediction"] = "A B C"

sample.to_csv("submission.csv", index=False)

sample.head()

In [ ]:
unique_options = pd.DataFrame({
    "Option": ["A", "B", "C", "D", "E"],
    "Unique Answer Choices": [train[col].nunique() for col in ["A", "B", "C", "D", "E"]]
})

unique_options

# Model 1 

# Baseline Model: TF-IDF + Cosine Similarity

In [28]:
# Important Libraries

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For checking the accuracy
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score)

In [7]:
#Logging into Weights and Biases

from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb.init(
    entity="25ds1000066-dl-genai-project",
    project="25ds1000066-t22026",
    name="Model1_TFIDF",
    config={
        "model": "TF-IDF",
        "similarity": "Cosine Similarity"
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [8]:
# Cleaning Data
import string
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
train_clean = train.copy()
test_clean = test.copy()

# Clean prompt
train_clean["prompt"] = train_clean["prompt"].apply(clean_text)
test_clean["prompt"] = test_clean["prompt"].apply(clean_text)

# Clean options
for option in ["A", "B", "C", "D", "E"]:
    train_clean[option] = train_clean[option].apply(clean_text)
    test_clean[option] = test_clean[option].apply(clean_text)

In [10]:
# Create combined Corpus
combined_text = (
    train_clean["prompt"] + " " +
    train_clean["A"] + " " +
    train_clean["B"] + " " +
    train_clean["C"] + " " +
    train_clean["D"] + " " +
    train_clean["E"]
)

In [11]:
# Fir TF IDF 

vectorizer = TfidfVectorizer(stop_words="english")

vectorizer.fit(combined_text)

TfidfVectorizer(stop_words='english')

In [4]:
# Creating MAP@3 Function

# Function to calculate Average Precision at K (AP@K) for a single prediction
def apk(actual, predicted, k=3):

    if len(predicted) > k:                   # Keep only the top-k predictions
        predicted = predicted[:k]            

    for i, p in enumerate(predicted):        # Iterate through the predicted labels
        if p == actual:                      # If the correct answer is found, return the score
            return 1 / (i + 1)

    return 0                                 # Return 0 if the correct answer is not in the top-k predictions


# Function to calculate Mean Average Precision at K (MAP@K)
def mapk(actuals, predictions, k=3):
    
    # Compute the average AP@K score over the entire dataset
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

In [13]:
# Generate Top-3 Predictions using TF-IDF Cosine Similarity

# Store the top-3 predictions for every question
predictions = []

# Store Top-1 predictions
tfidf_top1_predictions = []

for _, row in train_clean.iterrows():                             # Iterate through every question in the training dataset

    prompt_vec = vectorizer.transform([row["prompt"]])            # Convert the question prompt into a TF-IDF vector

    similarities = []                                             # Store cosine similarity scores for all answer options

    for option in ["A", "B", "C", "D", "E"]:                      # Compare the prompt with each of the five answer options

        option_vec = vectorizer.transform([row[option]])          # Convert the current answer option into a TF-IDF vector

        sim = cosine_similarity(prompt_vec, option_vec)[0][0]     # Calculate cosine similarity between the prompt and the answer option

        similarities.append((option, sim))                        # Save the option label together with its similarity score

    similarities.sort(key=lambda x: x[1], reverse=True)           # Sort answer options in descending order of cosine similarity

    top3 = [x[0] for x in similarities[:3]]                       # Select the three most similar answer options

   
    top1_tfidf = similarities[0][0]                               # Secleting the Top-1 prediction

    predictions.append(top3)                                      # Store the top-3 predictions for the current question

    tfidf_top1_predictions.append(top1_tfidf)                     # Store the top-1 predictions for the current question for checking the accuracy

In [14]:
# Evalutaion

tfidf_map3 = mapk(train["answer"], predictions)

print(f"TF-IDF Baseline MAP@3 : {tfidf_map3:.4f}")

TF-IDF Baseline MAP@3 : 0.2826


In [15]:
# Actual answers
actual_answers = train["answer"]

# Accuracy
tfidf_accuracy = accuracy_score(actual_answers,tfidf_top1_predictions)

# Precision
tfidf_precision = precision_score(actual_answers,tfidf_top1_predictions,average="macro")

# Recall
tfidf_recall = recall_score(actual_answers,tfidf_top1_predictions,average="macro")

# F1 Score
tfidf_f1 = f1_score(actual_answers,tfidf_top1_predictions,average="macro")

print(f"Accuracy : {tfidf_accuracy:.4f}")
print(f"Precision : {tfidf_precision:.4f}")
print(f"Recall : {tfidf_recall:.4f}")
print(f"F1 Score : {tfidf_f1:.4f}")

Accuracy : 0.1130
Precision : 0.1085
Recall : 0.1173
F1 Score : 0.1066


In [16]:
wandb.log({
    "Accuracy": tfidf_accuracy,
    "Precision": tfidf_precision,
    "Recall": tfidf_recall,
    "F1 Score": tfidf_f1,
    "MAP@3": tfidf_map3,
    "Kaggle Score": 0.30922
})

wandb.finish()

Accuracy,▁
F1 Score,▁
Kaggle Score,▁
MAP@3,▁
Precision,▁
Recall,▁
Accuracy,0.113
F1 Score,0.10655
Kaggle Score,0.30922
MAP@3,0.28258
Precision,0.10846


**Prediction on Test Data**

In [ ]:
# Generate Top-3 Predictions for the Test Dataset

tfidf_test_predictions = []

# Iterate through every question in the test dataset
for _, row in test_clean.iterrows():

    # Convert the question prompt into a TF-IDF vector
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    # Calculate cosine similarity between the prompt and each answer option
    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(prompt_vec, option_vec)[0][0]

        similarities.append((option, sim))

    # Sort options by similarity score (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Select the top 3 predicted answer labels
    top3 = [x[0] for x in similarities[:3]]

    tfidf_test_predictions.append(top3)

In [ ]:
# Create Submission File

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": [" ".join(pred) for pred in tfidf_test_predictions]
})

submission.head()

In [ ]:
# Save submission file
submission.to_csv("submission.csv", index=False)

In this baseline implementation, exploratory data analysis and text preprocessing were performed to better understand the dataset and prepare it for modeling. A classical Natural Language Processing approach using TF-IDF vectorization and cosine similarity was implemented to rank the five answer options for each multiple-choice question. The model was evaluated using the Mean Average Precision at 3 (MAP@3) metric, achieving a training MAP@3 score of 0.2962 and a Kaggle public leaderboard score of 0.30922. Although the TF-IDF baseline provides a simple and computationally efficient solution, it relies primarily on lexical similarity and does not capture contextual meaning or semantic relationships between the question prompt and answer options.

# Model 2

# Sentence Transformer (MiniLM)

Sentence Transformers generate dense semantic embeddings that capture the contextual meaning of entire sentences. Unlike TF-IDF, which relies on exact word overlap, Sentence Transformers understand semantic relationships between texts, making them more suitable for ranking multiple-choice answers based on meaning.

In [17]:
# W&B Login

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb.init(
    entity="25ds1000066-dl-genai-project",
    project="25ds1000066-t22026",
    name="Model2_MiniLM",
    config={
        "model": "MiniLM",
        "embedding_model": "all-MiniLM-L6-v2",
        "similarity": "Cosine Similarity"
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [18]:
# Loading the libraries
from sentence_transformers import SentenceTransformer, util

In [19]:
# Load pretrained MiniLM Sentence Transformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
# Generate Predictions

# Store Top-3 predictions
minilm_predictions = []

# Store top-1 prediction
minilm_top1_predictions = []

# Iterate through every question
for _, row in train.iterrows():

    # Encode prompt
    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    similarities = []

    # Compare prompt with each option
    for option in ["A", "B", "C", "D", "E"]:

        option_embedding = model.encode(
            row[option],
            convert_to_tensor=True
        )

        similarity = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()

        similarities.append((option, similarity))

    # Rank options
    similarities.sort(key=lambda x: x[1],reverse=True)
    
    
    # select top3 predictions
    top3_minlm_predictions = [x[0] for x in similarities[:3]]
    # select top1 predictions
    top1_minilm = similarities[0][0]


    #Store Predictions
    minilm_predictions.append(top3_minlm_predictions)
    minilm_top1_predictions.append(top1_minilm)

In [21]:
#Evaluation
minilm_map3 = mapk(train["answer"],minilm_predictions)

print(f"MiniLM MAP@3 : {minilm_map3:.4f}")

MiniLM MAP@3 : 0.4231


In [22]:
# Actual answers
actual_answers = train["answer"]

# Accuracy
minilm_accuracy = accuracy_score(actual_answers,minilm_top1_predictions)

# Precision
minilm_precision = precision_score(actual_answers,minilm_top1_predictions,average="macro")

# Recall
minilm_recall = recall_score(actual_answers,minilm_top1_predictions,average="macro")

# F1 Score
minilm_f1 = f1_score(actual_answers,minilm_top1_predictions,average="macro")

print(f"Accuracy : {minilm_accuracy:.4f}")
print(f"Precision : {minilm_precision:.4f}")
print(f"Recall : {minilm_recall:.4f}")
print(f"F1 Score : {minilm_f1:.4f}")

Accuracy : 0.2610
Precision : 0.2614
Recall : 0.2599
F1 Score : 0.2587


In [23]:
wandb.log({
    "Accuracy": minilm_accuracy,
    "Precision": minilm_precision,
    "Recall": minilm_recall,
    "F1 Score": minilm_f1,
    "MAP@3": minilm_map3,
    "Kaggle Score": 0.38653
})

wandb.finish()

Accuracy,▁
F1 Score,▁
Kaggle Score,▁
MAP@3,▁
Precision,▁
Recall,▁
Accuracy,0.261
F1 Score,0.25865
Kaggle Score,0.38653
MAP@3,0.42308
Precision,0.26136


**Prediction on Test Data**

In [ ]:
minilm_test_predictions = []

for _, row in test.iterrows():

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    similarities = []

    for option in ["A","B","C","D","E"]:

        option_embedding = model.encode(
            row[option],
            convert_to_tensor=True
        )

        similarity = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()

        similarities.append((option, similarity))

    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top3_test_minilm_predictions = [x[0] for x in similarities[:3]]

    minilm_test_predictions.append(" ".join(top3_test_minilm_predictions))

In [ ]:
submission = sample.copy()

submission["Prediction"] = minilm_test_predictions

submission.to_csv("submission.csv", index=False)

submission.head()

**Conclusion**

The Sentence Transformer (all-MiniLM-L6-v2) model demonstrated a substantial improvement over the TF-IDF baseline by utilizing contextual sentence embeddings rather than sparse lexical features. The model achieved a local MAP@3 score of 0.4231 and a Kaggle public leaderboard score of 0.38653, outperforming the TF-IDF baseline in both local evaluation and competition performance. This improvement indicates that semantic embeddings are more effective at capturing the contextual relationship between the question prompt and answer options. However, the results also suggest that there is still considerable room for improvement, motivating the exploration of more powerful embedding models and transformer-based architectures in subsequent experiments.

# Model 3

# Multilayer Perceptron 

In [7]:
# Deep Learning Libraries
import torch
import torch.nn as nn
import torch.optim as optim

# Dataset Utilities
from torch.utils.data import Dataset, DataLoader

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Others
import numpy as np
import pandas as pd

In [6]:
from kaggle_secrets import UserSecretsClient
import wandb


#Logging in Weights and Biases

user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

# Project details
wandb.init(
    entity="25ds1000066-dl-genai-project",
    project="25ds1000066-t22026",
    name="Model3_MLP_TFIDF",
    config={
        "model": "MLP",
        "input_features": "TF-IDF",
        "optimizer": "Adam",
        "loss_function": "CrossEntropyLoss"
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


This model is the first deep learning model developed completely from scratch for the Smart MCQ Solver Challenge. Instead of using cosine similarity to rank answers, a Multi-Layer Perceptron (MLP) is trained to learn patterns from TF-IDF features and predict the correct answer option. The model is implemented using PyTorch and trained using backpropagation and gradient descent.

In [8]:
# Split original questions
train_questions, valid_questions = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training Questions :", len(train_questions))
print("Validation Questions :", len(valid_questions))

Training Questions : 1600
Validation Questions : 400


In [10]:
# Create pairwise training data

def create_pairwise_dataset(dataframe):
# Create an empty list to store the new pairwise dataset
    pairwise_data = []

    for _, row in dataframe.iterrows():                                     # Loop through every question in the training dataset

        prompt = row["prompt"]                                          # Extract the question prompt
        correct_answer = row["answer"]                                  # Get the correct answer label (A, B, C, D or E)

        for option in ["A", "B", "C", "D", "E"]:                        # Loop through all five answer options
        
        # Create one training sample consisting of:Question + one answer option
            pairwise_data.append({
                "text": prompt + " " + row[option],                    # Combine question and option into one text input

    
                # Assign label:
                # 1 = Correct option
                # 0 = Incorrect option
                "label": 1 if option == correct_answer else 0
        })

    return pd.DataFrame(pairwise_data)


# Convert the list into a Pandas DataFrame
pairwise_train = create_pairwise_dataset(train_questions)

pairwise_valid = create_pairwise_dataset(valid_questions)

print(pairwise_train.shape)
print(pairwise_valid.shape)

(8000, 2)
(2000, 2)


The original dataset contains one row per question with five answer options. Since our MLP is designed as a binary classifier (Correct/Incorrect), we transform each question into five separate training samples by pairing the prompt with each answer option. The correct option receives a label of 1, while the remaining options receive 0.

In [13]:
# Create TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Fit only on training data
X_train = vectorizer.fit_transform(pairwise_train["text"])

# Transform validation data
X_valid = vectorizer.transform(pairwise_valid["text"])

# Labels
y_train = pairwise_train["label"].values
y_valid = pairwise_valid["label"].values

In [14]:
# Convert sparse matrices to dense arrays
X_train = X_train.toarray()
X_valid = X_valid.toarray()

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_valid_tensor = torch.FloatTensor(X_valid)

y_train_tensor = torch.LongTensor(y_train)
y_valid_tensor = torch.LongTensor(y_valid)

print("Training Tensor Shape :", X_train_tensor.shape)
print("Validation Tensor Shape :", X_valid_tensor.shape)

Training Tensor Shape : torch.Size([8000, 2761])
Validation Tensor Shape : torch.Size([2000, 2761])


In [15]:
# Custom Dataset

# Define a custom dataset class
class MCQDataset(Dataset):

    def __init__(self, features, labels):                      # Initialize the dataset with features and labels

        self.features = features                               # Store feature vectors
        self.labels = labels                                   # Store corresponding labels

    def __len__(self):                                         # function for Return the total number of samples

        return len(self.labels)
    
    def __getitem__(self, index):                              # function for Return one sample (feature, label) at the given index

        return self.features[index], self.labels[index]

PyTorch trains models using Dataset objects. This custom dataset stores the feature vectors and labels together, allowing the DataLoader to efficiently load batches of training data during model training

In [16]:
# Creating Dataset objects

# Create dataset object for training data
train_dataset = MCQDataset(
    X_train_tensor,
    y_train_tensor
)

# Create dataset object for validation data
valid_dataset = MCQDataset(
    X_valid_tensor,
    y_valid_tensor
)

After creating the custom Dataset class, we create separate dataset objects for the training and validation sets. These objects will later be passed to the DataLoader.

In [17]:
# Creating Dataloaders

# Create DataLoader for training
train_loader = DataLoader(
    train_dataset,
    batch_size=64,                  # Number of samples processed at a time
    shuffle=True                    # Shuffle data every epoch to improve learning
)
# Create DataLoader for validation
valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False                    # Keep validation order fixed
)

The DataLoader loads the dataset in small batches instead of loading the entire dataset at once. This improves memory efficiency and speeds up neural network training.

In [18]:
#Defining Model

# Multi-Layer Perceptron Model
class MLPClassifier(nn.Module):

    def __init__(self, input_size):            # Initialize the neural network

        super().__init__()                     # Call the parent class constructor

        self.network = nn.Sequential(          # Build the neural network

            nn.Linear(input_size, 512),        # First hidden layer
            nn.ReLU(),                         # Activation function
            nn.Dropout(0.3),                   # Dropout layer to reduce overfitting

            nn.Linear(512, 128),               # Second hidden layer
            nn.ReLU(),                         # Activation function
            nn.Dropout(0.3),                   # Dropout layer
            
            
            # Output layer
            # Produces scores for two classes:
            # 0 = Incorrect
            # 1 = Correct
            nn.Linear(128, 2)                  

        )
    # Forward pass through the network
    def forward(self, x):

        return self.network(x)

This defines the Multi-Layer Perceptron (MLP) architecture. The model learns patterns from TF-IDF features and predicts whether a given question-option pair is correct (1) or incorrect (0).

In [19]:
# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
mlp_model = MLPClassifier(
    input_size=X_train.shape[1]
).to(device)

print(mlp_model)

MLPClassifier(
  (network): Sequential(
    (0): Linear(in_features=2761, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [20]:
#Defining loss function

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

In [22]:
# Training Parameters
num_epochs = 10

# Train the model
for epoch in range(num_epochs):

    mlp_model.train()

    running_loss = 0
    correct = 0
    total = 0

    for features, labels in train_loader:

        # Move data to GPU
        features = features.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = mlp_model(features)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        # Calculate training accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct / total

    # Log metrics to W&B
    wandb.log({
        "Epoch": epoch + 1,
        "Training Loss": epoch_loss,
        "Training Accuracy": epoch_accuracy
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.4f}"
    )

Epoch [1/10] Loss: 0.0618 Accuracy: 0.9685
Epoch [2/10] Loss: 0.0619 Accuracy: 0.9670
Epoch [3/10] Loss: 0.0616 Accuracy: 0.9670
Epoch [4/10] Loss: 0.0583 Accuracy: 0.9691
Epoch [5/10] Loss: 0.0589 Accuracy: 0.9704
Epoch [6/10] Loss: 0.0575 Accuracy: 0.9702
Epoch [7/10] Loss: 0.0558 Accuracy: 0.9695
Epoch [8/10] Loss: 0.0547 Accuracy: 0.9700
Epoch [9/10] Loss: 0.0530 Accuracy: 0.9709
Epoch [10/10] Loss: 0.0557 Accuracy: 0.9709


Before training the neural network, we define the loss function, optimizer, and training parameters. The loss function measures the prediction error, while the optimizer updates the model weights using backpropagation. The training loop repeatedly processes batches of data over multiple epochs, allowing the model to learn patterns from the training dataset. Training loss and accuracy are also logged to Weights & Biases (W&B) for experiment tracking.

In [23]:
# Validation

# Set the model to evaluation mode
mlp_model.eval()

valid_loss = 0                                         # Initialize validation loss
correct = 0                                            # Initialize correct prediction counter
total = 0                                              # Initialize total sample counter

with torch.no_grad():                                  # Disable gradient calculation during validation

    for features, labels in valid_loader:              # Loop through every validation batch

        features = features.to(device)                 # Move data to GPU (or CPU)
        labels = labels.to(device)

        outputs = mlp_model(features)                  # Forward pass

        loss = criterion(outputs, labels)              # Compute validation loss

        valid_loss += loss.item()                      # Accumulate batch loss

        _, predicted = torch.max(outputs, 1)           # Get predicted class

        total += labels.size(0)                        # Count total validation samples

        correct += (predicted == labels).sum().item()  # Count correctly classified samples

validation_loss = valid_loss / len(valid_loader)       # Calculate average validation loss
validation_accuracy = correct / total                  # Calculate validation accuracy

print(f"Validation Loss : {validation_loss:.4f}")
print(f"Validation Accuracy : {validation_accuracy:.4f}")

# Log validation metrics to Weights & Biases
wandb.log({
    "Validation Loss": validation_loss,
    "Validation Accuracy": validation_accuracy
})

Validation Loss : 0.0505
Validation Accuracy : 0.9715


After training, we evaluate the model on the validation dataset to measure how well it performs on unseen data. Validation loss and accuracy help determine whether the model has learned useful patterns or has overfitted the training data.

In [24]:

mlp_model.eval()

# To store Top-3 predictions
mlp_predictions = []

# To store top 1 predictions
mlp_top1_predictions = []

# Disable gradient calculation
with torch.no_grad():

    # Loop through every question
    for _, row in valid_questions.iterrows():

        option_scores = []

        # Evaluate each option
        for option in ["A", "B", "C", "D", "E"]:

            # Combine prompt and option
            text = row["prompt"] + " " + row[option]

            # Convert to TF-IDF
            text_vector = vectorizer.transform([text]).toarray()

            # Convert to tensor
            text_tensor = torch.FloatTensor(text_vector).to(device)

            # Model prediction
            output = mlp_model(text_tensor)

            # Probability of class = Correct (label 1)
            probability = torch.softmax(output, dim=1)[0][1].item()

            option_scores.append((option, probability))

        # Sort by probability
        option_scores.sort(
            key=lambda x: x[1],
            reverse=True
        )

        # Seleting Top-3 predictions
        top3_mlp = [x[0] for x in option_scores[:3]]

        # Selecting Top-1 prediction
        top1_mlp = option_scores[0][0]

        # Store Predictions
        mlp_predictions.append(top3_mlp)
        mlp_top1_predictions.append(top1_mlp)

In [26]:
#Evaluting on MAP@3

mlp_map3 = mapk(valid_questions["answer"],mlp_predictions)

print(f"MLP MAP@3 : {mlp_map3:.4f}")

MLP MAP@3 : 0.9654


In [29]:
# Actual answers
actual_answers = valid_questions["answer"]

# Accuracy
mlp_accuracy = accuracy_score(actual_answers,mlp_top1_predictions)

# Precision
mlp_precision = precision_score(actual_answers,mlp_top1_predictions,average="macro")

# Recall
mlp_recall = recall_score(actual_answers,mlp_top1_predictions,average="macro")

# F1 Score
mlp_f1 = f1_score(actual_answers,mlp_top1_predictions,average="macro")

print(f"Accuracy : {mlp_accuracy:.4f}")
print(f"Precision : {mlp_precision:.4f}")
print(f"Recall : {mlp_recall:.4f}")
print(f"F1 Score : {mlp_f1:.4f}")

Accuracy : 0.9375
Precision : 0.9433
Recall : 0.9426
F1 Score : 0.9397


In [30]:
# Storing thr prdiction in W&B

wandb.log({

    "Accuracy": mlp_accuracy,

    "Precision": mlp_precision,

    "Recall": mlp_recall,

    "F1 Score": mlp_f1,

    "MAP@3": mlp_map3,

    "Kaggle Score": 0.72901

})

wandb.finish()

Accuracy,▁
Epoch,▁▂▃▃▄▅▆▆▇█▁▂▃▃▄▅▆▆▇█
F1 Score,▁
Kaggle Score,▁
MAP@3,▁
Precision,▁
Recall,▁
Training Accuracy,▁▄▆▇▇▇██████████████
Training Loss,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁
+1,...


After training the model, we use it to predict the probability that each answer option is correct. For every question, all five options are evaluated individually, and their probabilities are ranked in descending order. The top three highest-scoring options are selected as the final predictions, which are later used to calculate the MAP@3 score and generate the Kaggle submission.

In [ ]:
# Store test predictions
mlp_test_predictions = []

mlp_model.eval()

with torch.no_grad():

    for _, row in test.iterrows():

        option_scores = []

        for option in ["A", "B", "C", "D", "E"]:

            text = row["prompt"] + " " + row[option]

            text_vector = vectorizer.transform([text]).toarray()

            text_tensor = torch.FloatTensor(text_vector).to(device)

            output = mlp_model(text_tensor)

            probability = torch.softmax(output, dim=1)[0][1].item()

            option_scores.append((option, probability))

        option_scores.sort(
            key=lambda x: x[1],
            reverse=True
        )

        top3_mlp = [x[0] for x in option_scores[:3]]

        mlp_test_predictions.append(" ".join(top3_mlp))

In [ ]:
submission = sample.copy()

submission["Prediction"] = mlp_test_predictions

submission.to_csv("submission.csv", index=False)

submission.head()

The Multi-Layer Perceptron (MLP) was the first deep learning model developed completely from scratch for this project. Unlike the previous models, the MLP learned to classify each question-option pair as either correct or incorrect using TF-IDF features. The model was implemented using PyTorch and trained with backpropagation, achieving a validation accuracy of **96.60%** and a **Kaggle score of 0.72901**, which was a significant improvement over both the TF-IDF and MiniLM baseline models. This model also introduced GPU training and experiment tracking using Weights & Biases (W&B), providing a strong foundation for building and fine-tuning more advanced transformer-based models in the next stage of the project.

# Model 4

# BGE Embeddings with Multi-Layer Perceptron (MLP)

After observing that Model 3 (TF-IDF + MLP) significantly outperformed the embedding-based cosine similarity models, the next objective was to investigate whether replacing traditional TF-IDF features with richer semantic embeddings could further improve performance.

In this model, the BAAI/bge-base-en-v1.5 sentence embedding model was used to generate dense vector representations for each Question–Option pair. These embeddings capture the semantic meaning of the text and provide a more informative representation than sparse TF-IDF vectors.

The generated 768-dimensional embeddings were then used as inputs to a custom-built Multi-Layer Perceptron (MLP) classifier implemented in PyTorch. The classifier predicts the probability of each answer option being correct, after which the five options corresponding to a question are ranked by probability, and the top three predictions are selected to compute the MAP@3 score.

This model combines the strengths of pretrained transformer-based embeddings with a lightweight neural network classifier while avoiding the computational complexity of fine-tuning large transformer models.

In [5]:
!pip install -q sentence-transformers wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

In [31]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from sentence_transformers import SentenceTransformer

from kaggle_secrets import UserSecretsClient
import wandb

**** Weights & Biases Logging in****

In [34]:
# Get API key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login(key=wandb_api)

# Start a new run
wandb.init(
    project="25ds1000066-t22026",
    name="Model4_BGE_MLP_v1",
    config={
        "Embedding Model": "BAAI/bge-base-en-v1.5",
        "Embedding Dimension": 768,
        "Architecture": "768-512-128-2",
        "Dropout": 0.3,
        "Batch Size": 64,
        "Epochs": 20,
        "Learning Rate": 5e-4,
        "Optimizer": "AdamW",
        "Weight Decay": 1e-4,
        "Loss Function": "Weighted CrossEntropyLoss",
        "Scheduler": "ReduceLROnPlateau",
        "Normalize Embeddings": True
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [35]:
# Train Validation split

train_questions, valid_questions = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["answer"]
)

print("Training Questions:", train_questions.shape)
print("Validation Questions:", valid_questions.shape)

Training Questions: (1600, 8)
Validation Questions: (400, 8)


The dataset is divided into training and validation sets using an 80:20 ratio. The split is performed at the question level rather than the pairwise level to prevent data leakage. Stratified sampling ensures that the distribution of answer classes remains similar in both subsets

In [37]:
# Question +Option datset

def create_embedding_dataset(df, is_train=True):
    
    texts = []                                                   # Stores the combined Question + Option text
    
    labels = []                                                  # Stores binary labels (only for training data)
    
    option_columns = ["A", "B", "C", "D", "E"]                   # Available answer options
    
    for _, row in df.iterrows():                                # Iterate through every question
        
        question = row["prompt"]                                # Extract the question
        
        for option in option_columns:                           # Create one sample for each option

            
            text = f"Question: {question} [SEP] Option: {row[option]}"     # Combine Question and Option into a single sentence
            
            texts.append(text)                                 # Save the combined text
            
            if is_train:                                       # Create labels only for training data
                
                labels.append(1 if option == row["answer"] else 0)         # Correct option = 1, Incorrect option = 0

    # Return texts and labels for training
    if is_train:
        return texts, labels

    # Return only texts for test data
    else:
        return texts

Each question is converted into five Question–Option pairs. During training, the correct option is assigned label 1, while the remaining four options receive label 0. This transforms the multi-class problem into a binary classification problem for the MLP.

In [38]:
# Generating Training and validation data

# Generate Question-Option pairs for training
embedding_train, y_train = create_embedding_dataset(train_questions)

# Generate Question-Option pairs for validation
embedding_valid, y_valid = create_embedding_dataset(valid_questions)

print("Training Samples :", len(embedding_train))
print("Validation Samples:", len(embedding_valid))

print("\nFirst Sample:")
print(embedding_train[0])

print("\nFirst Label:")
print(y_train[0])

Training Samples : 8000
Validation Samples: 2000

First Sample:
Question: Select the most accurate option: What is Modified Newtonian Dynamics (MOND)? from the following choices. [SEP] Option: MOND is a principle that explains the behavior of light in the presence of strong gravitational fields. It is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics.

First Label:
0


The previously defined function is applied separately to the training and validation datasets. This creates the Question–Option pairs and their corresponding binary labels required for model training and evaluation.

In [39]:
# Load BGE Embedding Model

bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

print("BGE model loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE model loaded successfully!


The pretrained BAAI/bge-base-en-v1.5 model is loaded using the Sentence Transformers library. This model converts each Question–Option pair into a dense 768-dimensional semantic embedding, which serves as the input features for the MLP classifier.

In [40]:
#Generate BGE Embeddings

# Generate embeddings for all training Question-Option pairs
print("Generating training embeddings...")

bge_train_embeddings = bge_model.encode(
    embedding_train,                            # Training text samples
    batch_size=64,                              # Process 64 samples at a time
    show_progress_bar=True,                     # Display embedding progress
    convert_to_numpy=True,                      # Convert embeddings to NumPy arrays
    normalize_embeddings=True                   # Normalize embedding vectors
)

# Generate embeddings for validation Question-Option pairs
print("Generating validation embeddings...")

bge_valid_embeddings = bge_model.encode(
    embedding_valid,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nTraining Embeddings Shape:", bge_train_embeddings.shape)
print("Validation Embeddings Shape:", bge_valid_embeddings.shape)

Generating training embeddings...


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Generating validation embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Training Embeddings Shape: (8000, 768)
Validation Embeddings Shape: (2000, 768)


The pretrained BGE model is used to convert each Question–Option pair into a dense 768-dimensional semantic embedding. These embeddings capture the contextual meaning of the text and serve as numerical feature vectors for training the neural network.

In [41]:
#Saving the embeddings
np.save("bge_train_embeddings.npy", bge_train_embeddings)
np.save("bge_valid_embeddings.npy", bge_valid_embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [42]:
# Convert embeddings vectors to PyTorch tensors
X_train = torch.tensor(bge_train_embeddings, dtype=torch.float32)
X_valid = torch.tensor(bge_valid_embeddings, dtype=torch.float32)

# Convert labels into integer tensors
y_train = torch.tensor(y_train, dtype=torch.long)
y_valid = torch.tensor(y_valid, dtype=torch.long)

print("X_train shape :", X_train.shape)
print("X_valid shape :", X_valid.shape)

print("y_train shape :", y_train.shape)
print("y_valid shape :", y_valid.shape)

X_train shape : torch.Size([8000, 768])
X_valid shape : torch.Size([2000, 768])
y_train shape : torch.Size([8000])
y_valid shape : torch.Size([2000])


The generated NumPy embeddings and labels are converted into PyTorch tensors. This enables efficient computation and GPU acceleration during neural network training.

In [43]:
# Defining BGE + MLP model

class BGEMLP(nn.Module):

    def __init__(self):                            # Initialize the neural network architecture
        super(BGEMLP, self).__init__()

        self.network = nn.Sequential(              # Sequential neural network

            nn.Linear(768, 512),                   # First hidden layer
            nn.ReLU(),                             # Activation function
            nn.Dropout(0.3),                       # Reduce overfitting

            nn.Linear(512, 128),                   # Second hidden layer
            nn.ReLU(),                             # Activation function
            nn.Dropout(0.3),                       # Dropout layer

            nn.Linear(128, 2)                      # Output layer (Binary Classification)

        )

    # Forward propagation
    def forward(self, x):
        return self.network(x)

A Multi-Layer Perceptron (MLP) is designed to classify each Question–Option embedding as either correct or incorrect. The network consists of two hidden layers with ReLU activation and Dropout regularization to reduce overfitting.

In [44]:
# Creating model, loss function, optimizer

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Initialize model
mlp_model = BGEMLP().to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    mlp_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

# Sheduler
# Reduce learning rate if validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("\nModel Summary:")
print(mlp_model)

Using device: cuda

Model Summary:
BGEMLP(
  (network): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [45]:
# Creating Dataset Class
class EmbeddingDataset(Dataset):

    # Store embeddings and labels
    def __init__(self, X, y):
        self.X = X
        self.y = y

    # Return total number of samples
    def __len__(self):
        return len(self.X)

    # Retrieve one sample at a time
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [46]:
# Creating Dataloaders

# Training DataLoader (shuffle enabled)
train_loader = DataLoader(
    EmbeddingDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

# Validation DataLoader (shuffle disabled)
valid_loader = DataLoader(
    EmbeddingDataset(X_valid, y_valid),
    batch_size=64,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(valid_loader))

Train batches: 125
Validation batches: 32


A custom PyTorch Dataset is created to efficiently manage embeddings and labels. DataLoaders divide the dataset into mini-batches, enabling faster and memory-efficient model training.

In [47]:
# Weighted Loss function

# Class weights
# Assign higher weight to the minority class
class_weights = torch.tensor([1.0, 4.0], dtype=torch.float32).to(device)

# Weighted CrossEntropy Loss
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Using weighted CrossEntropyLoss")
print("Class Weights:", class_weights)

Using weighted CrossEntropyLoss
Class Weights: tensor([1., 4.], device='cuda:0')


Since each question has one correct answer and four incorrect answers, the dataset is imbalanced. Weighted CrossEntropyLoss assigns a larger penalty to incorrectly predicting the correct option, encouraging the model to focus more on minority-class samples.

In [48]:
# Training Loop
 
epochs = 20                                                 # Number of complete passes through the training dataset                                           

train_losses = []                                           # Lists to store training and validation loss after each epoch
valid_losses = []

# Initialize the best validation loss with infinity
# This helps identify and save the best-performing model
best_valid_loss = float("inf")

for epoch in range(epochs):                                  # Iterate through each training epoch

  
    # Training Phase
   
    mlp_model.train()                                      # Set the model to training mode & Enables Dropout and gradient computation

    running_train_loss = 0.0                               # Variable to accumulate training loss for the current epoch

    for X_batch, y_batch in train_loader:                  # Iterate through each mini-batch in the training data

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()                             # Clear gradients from the previous iteration

        outputs = mlp_model(X_batch)                      # Perform a forward pass through the neural network

        loss = criterion(outputs, y_batch)                # Compute the classification loss

        loss.backward()                                   # Perform backpropagation to compute gradients

        optimizer.step()                                  # Update the model parameters

        running_train_loss += loss.item()                   # Accumulate batch loss

     # Compute the average training loss for the current epoch
    avg_train_loss = running_train_loss / len(train_loader)

    
    # Validation
    
    mlp_model.eval()                                    # Set the model to evaluation mode & Disables Dropout and gradient updates

    running_valid_loss = 0.0                            # Variable to accumulate validation loss

    with torch.no_grad():                               # Disable gradient computation during validation

        for X_batch, y_batch in valid_loader:           # Iterate through validation batches

            X_batch = X_batch.to(device)                # Move validation data to the selected device
            y_batch = y_batch.to(device)

            outputs = mlp_model(X_batch)                # Forward pass

            loss = criterion(outputs, y_batch)          # Compute validation loss

            running_valid_loss += loss.item()           # Accumulate validation loss

    avg_valid_loss = running_valid_loss / len(valid_loader)        # Compute the average validation loss
    
      
    train_losses.append(avg_train_loss)                # Store losses for visualization or later analysis
    valid_losses.append(avg_valid_loss)

    
    # Save Best Model
    
    # Save the model if validation loss improves
    if avg_valid_loss < best_valid_loss:

        # Update the best validation loss
        best_valid_loss = avg_valid_loss

        # Save the model parameters
        torch.save(
            mlp_model.state_dict(),
            "best_bge_mlp_model.pth"
        )

        print("Best model saved!")

    # Update learning rate scheduler
    # Learning rate is reduced if validation loss stops improving
    scheduler.step(avg_valid_loss)

    
    # Print Progress
    
    print(
    f"Epoch {epoch+1:02d}/{epochs} | "
    f"Train Loss: {avg_train_loss:.4f} | "
    f"Valid Loss: {avg_valid_loss:.4f} | "
    f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        

    
    # W&B Logging
    
    wandb.log({
        "Epoch": epoch + 1,
        "Train Loss": avg_train_loss,
        "Best Validation Loss": best_valid_loss,
        "Learning Rate": optimizer.param_groups[0]["lr"]
    })

print("\nTraining Completed!")

print(f"Best Validation Loss : {best_valid_loss:.4f}")

Best model saved!
Epoch 01/20 | Train Loss: 0.6932 | Valid Loss: 0.6919 | LR: 0.000500
Best model saved!
Epoch 02/20 | Train Loss: 0.6906 | Valid Loss: 0.6868 | LR: 0.000500
Best model saved!
Epoch 03/20 | Train Loss: 0.6853 | Valid Loss: 0.6809 | LR: 0.000500
Best model saved!
Epoch 04/20 | Train Loss: 0.6757 | Valid Loss: 0.6599 | LR: 0.000500
Best model saved!
Epoch 05/20 | Train Loss: 0.6581 | Valid Loss: 0.6400 | LR: 0.000500
Best model saved!
Epoch 06/20 | Train Loss: 0.6405 | Valid Loss: 0.6093 | LR: 0.000500
Best model saved!
Epoch 07/20 | Train Loss: 0.6133 | Valid Loss: 0.5785 | LR: 0.000500
Best model saved!
Epoch 08/20 | Train Loss: 0.5701 | Valid Loss: 0.5383 | LR: 0.000500
Best model saved!
Epoch 09/20 | Train Loss: 0.5364 | Valid Loss: 0.5335 | LR: 0.000500
Best model saved!
Epoch 10/20 | Train Loss: 0.4951 | Valid Loss: 0.4581 | LR: 0.000500
Best model saved!
Epoch 11/20 | Train Loss: 0.4626 | Valid Loss: 0.4174 | LR: 0.000500
Best model saved!
Epoch 12/20 | Train Loss:

The model is trained for 20 epochs using mini-batch gradient descent. After each epoch, the validation loss is computed, the best-performing model is saved, the learning rate scheduler is updated, and training statistics are logged to Weights & Biases (W&B).

In [51]:
# Load Best Saved Model

mlp_model.load_state_dict(
    torch.load(
        "best_bge_mlp_model.pth",
        map_location=device
    )
)

mlp_model.eval()

# Validation Prediction

validation_probabilities = []                   # Store probability of the positive class (Correct Answer)
validation_predictions = []                     # Store predicted class labels (0 or 1)
validation_labels = []                          # Store actual class labels

with torch.no_grad():                                     # Disable gradient computation during inference

    for X_batch, y_batch in valid_loader:                 # Iterate through validation batches

        X_batch = X_batch.to(device)                      # Move input embeddings to the selected device

        outputs = mlp_model(X_batch)                      # Forward pass through the trained model

        probabilities = torch.softmax(outputs, dim=1)     # Convert logits into probabilities

        preds = outputs.argmax(dim=1)                     # Select the class with the highest probability

        validation_probabilities.extend(                  # Store probability of the positive class (Correct Answer)
            probabilities[:, 1].cpu().numpy()
        )

        validation_predictions.extend(                    # Store predicted labels
            preds.cpu().numpy()
        )

        validation_labels.extend(                         # Store true labels
            y_batch.numpy()
        )


After training, the best-performing model (based on the lowest validation loss) is reloaded. The model then predicts the probability of each Question–Option pair in the validation dataset. These probabilities are later used to rank the five answer options and compute the MAP@3 evaluation metric.

In [52]:
# Accuracy & F1

accuracy = accuracy_score(
    validation_labels,
    validation_predictions
)

f1 = f1_score(
    validation_labels,
    validation_predictions
)

print(f"Validation Accuracy : {accuracy:.4f}")
print(f"Validation F1 Score : {f1:.4f}")


Validation Accuracy : 0.9085
Validation F1 Score : 0.7955


In [53]:
# MAP@3 Function

def mapk(actual, predicted, k=3):

    score = 0.0

    for a, p in zip(actual, predicted):

        try:
            index = p[:k].index(a)
            score += 1.0 / (index + 1)

        except ValueError:
            pass

    return score / len(actual)

In [54]:
# Group Predictions into Questions

option_labels = ["A", "B", "C", "D", "E"]              # List of answer option labels

validation_top3_predictions = []                       # Store Top-3 predicted options for each question
validation_actual_answers = []                         # Store actual correct answers

probability_index = 0                                  # Keeps track of the current prediction index

for _, row in valid_questions.iterrows():              # Iterate through each validation question

    option_scores = []                                 # Store probability for each answer option

    for option in option_labels:                       # Retrieve probabilities of the five options

        option_scores.append(
            (
                option,                               # Store option label and its predicted probability
                validation_probabilities[probability_index]
            )
        )

        probability_index += 1                        # Move to the next probability

    option_scores = sorted(                           # Sort options by predicted probability (highest first)
        option_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in option_scores[:3]]          # Select the three most probable options

    validation_top3_predictions.append(top3)          # Store Top-3 predictions

    validation_actual_answers.append(row["answer"])   # Store the actual correct answer

Since the model predicts probabilities for individual Question–Option pairs, the predictions must be grouped back into their original questions. The five option probabilities are ranked in descending order, and the top three options are selected for computing the MAP@3 score.

In [55]:
# Validation MAP@3

validation_map3 = mapk(
    validation_actual_answers,
    validation_top3_predictions,
    k=3
)

print(f"Validation MAP@3 : {validation_map3:.5f}")

Validation MAP@3 : 0.92917


In [56]:
# Log Metrics to W&B

wandb.log({
    "Validation Accuracy": accuracy,
    "Validation F1": f1,
    "Validation MAP@3": validation_map3
})

# Save Model Artifact
artifact = wandb.Artifact(
    "bge_mlp_model",
    type="model"
)

artifact.add_file(
    "best_bge_mlp_model.pth"
)

wandb.log_artifact(artifact)

wandb.finish()

Best Validation Loss,███▇▇▇▆▆▆▄▄▃▃▃▃▂▂▂▁▁
Epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
Learning Rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Train Loss,████▇▇▇▆▅▅▄▄▃▃▂▂▂▂▁▁
Validation Accuracy,▁
Validation F1,▁
Validation MAP@3,▁
Best Validation Loss,0.24669
Epoch,20
Learning Rate,0.0005
Train Loss,0.29224


In [57]:
torch.save({
    "epoch": epoch,
    "model_state_dict": mlp_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_valid_loss": best_valid_loss,
}, "best_checkpoint.pth")

**Submission on Test data**

In [58]:
# Create Test Question + Option Text

embedding_test = create_embedding_dataset(
    test,
    is_train=False
)

print("Test Samples:", len(embedding_test))
print("\nFirst Test Sample:")
print(embedding_test[0])

Test Samples: 2500

First Test Sample:
Question: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. [SEP] Option: For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy.


In [59]:
# Generate BGE embeddings for test data

print("Generating test embeddings...")

bge_test_embeddings = bge_model.encode(
    embedding_test,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Test Embeddings Shape:", bge_test_embeddings.shape)

Generating test embeddings...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Test Embeddings Shape: (2500, 768)


In [60]:
# Convert to tensor
X_test = torch.tensor(
    bge_test_embeddings,
    dtype=torch.float32
)

In [61]:
# Test Dataloader
test_loader = DataLoader(
    X_test,
    batch_size=64,
    shuffle=False
)

In [62]:
# Load the best model

mlp_model.load_state_dict(
    torch.load(
        "best_bge_mlp_model.pth",
        map_location=device
    )
)

mlp_model.eval()

print("Best model loaded successfully!")

Best model loaded successfully!


In [63]:
# Predict the test probabilities

test_probabilities = []

with torch.no_grad():

    for X_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = mlp_model(X_batch)

        probabilities = torch.softmax(outputs, dim=1)

        test_probabilities.extend(
            probabilities[:, 1].cpu().tolist()
        )

print("Total probabilities:", len(test_probabilities))

Total probabilities: 2500


In [64]:
# Covert to top3 predictions

option_labels = ["A", "B", "C", "D", "E"]

bge_predictions = []

probability_index = 0

for _, row in test.iterrows():

    option_scores = []

    for option in option_labels:

        option_scores.append(
            (
                option,
                test_probabilities[probability_index]
            )
        )

        probability_index += 1

    option_scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [option for option, _ in option_scores[:3]]

    bge_predictions.append(" ".join(top3))

print(bge_predictions[:5])

['A E B', 'B C E', 'B D E', 'E C A', 'C A D']


In [65]:
submission = sample.copy()

submission["Prediction"] = bge_predictions

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A E B
1,2,B C E
2,3,B D E
3,4,E C A
4,5,C A D


Although the BGE embeddings produced strong validation performance, the public Kaggle score (0.70074) did not surpass the TF-IDF + MLP baseline (0.72901). This indicates that while pretrained semantic embeddings captured meaningful contextual information, they did not generalize as effectively to the hidden evaluation set under the current pairwise classification framework.

The results suggest that the overall prediction strategy and ranking mechanism play a more significant role than simply replacing TF-IDF with dense sentence embeddings. Future work could explore alternative architectures, such as transformer feature extraction with different pooling strategies, sentence-pair representations, or ensemble methods that combine the strengths of multiple models.

# Miscellaneous

****MileStone 1****

Q1,Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option? 

In [ ]:
import pandas as pd

# Load the training dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Calculate the frequency of each correct answer
answer_counts = train["answer"].value_counts().sort_index()

print("Frequency Distribution:")
print(answer_counts)

# Find the most and least frequent counts
most_frequent = answer_counts.max()
least_frequent = answer_counts.min()

# Calculate their sum
result = most_frequent + least_frequent

print("\nMost Frequent Count :", most_frequent)
print("Least Frequent Count:", least_frequent)
print("Sum =", result)

Q2, After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [ ]:
import string


# Create a translation table to remove punctuation
translator = str.maketrans("", "", string.punctuation)

# Clean the prompt column:
# 1. Convert to lowercase
# 2. Remove punctuation
# 3. Split into words
cleaned_words = (
    train["prompt"]
    .astype(str)
    .str.lower()
    .str.translate(translator)
    .str.split()
)

# Build the vocabulary (unique words)
vocabulary = set()

for words in cleaned_words:
    vocabulary.update(words)

# Vocabulary size
print("Vocabulary Size:", len(vocabulary))

Q3, Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Select Row ID = 1
prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# Clean the text
cleaned = prompt.lower().translate(translator)

# Split into words
words = cleaned.split()

# Remove English stop words
filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

# Print results
print("Original Prompt:")
print(prompt)

print("\nWords after removing stop words:")
print(filtered_words)

print("\nNumber of words left:", len(filtered_words))

Q4, Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine prompt and all answer options into one text per row
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# Fit the vectorizer
X = vectorizer.fit_transform(combined_text)

# Vocabulary size (number of feature columns)
print("Total feature columns:", len(vectorizer.get_feature_names_out()))

# Alternative (same answer)
print("Shape:", X.shape)
print("Number of features:", X.shape[1])

Q5, Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Select Row ID = 1
row = train.loc[train["id"] == 1].iloc[0]

# Transform the prompt and Option A using the already fitted vectorizer
prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

# Calculate cosine similarity
similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print("Cosine Similarity:", round(similarity, 4))

Q6, Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer. 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

correct_predictions = 0

# Loop through every row
for _, row in train.iterrows():

    # TF-IDF vector of the prompt
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    # Calculate similarity with each option
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities.append(sim)

    # Find the option with the highest similarity
    predicted_option = ["A", "B", "C", "D", "E"][similarities.index(max(similarities))]

    # Compare with the actual answer
    if predicted_option == row["answer"]:
        correct_predictions += 1

# Calculate percentage accuracy
percentage = (correct_predictions / len(train)) * 100

print(f"Correct Predictions : {correct_predictions}")
print(f"Total Questions     : {len(train)}")
print(f"Percentage          : {percentage:.2f}%")

Q7, If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

1.0

Q8, If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E? 

0.5

Q9, The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [ ]:
# Top 3 most frequent answer labels
top3 = answer_counts.index[:3].tolist()

print("Top 3 Majority Classes:", top3)

# Calculate MAP@3
map3_score = 0

for actual in train["answer"]:
    if actual == top3[0]:
        map3_score += 1.0
    elif actual == top3[1]:
        map3_score += 0.5
    elif actual == top3[2]:
        map3_score += 1/3
    else:
        map3_score += 0

map3_score /= len(train)

print(f"Majority Class Baseline MAP@3: {map3_score:.4f}")

Q10, The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

map3_score = 0

# Loop through each question
for _, row in train.iterrows():

    # TF-IDF vector of the prompt
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    # Compute similarity with each option
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities.append((option, sim))

    # Sort options by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Top 3 predicted options
    top3_predictions = [x[0] for x in similarities[:3]]

    # Ground truth
    actual = row["answer"]

    # Calculate AP@3 for this row
    if actual in top3_predictions:
        rank = top3_predictions.index(actual) + 1
        map3_score += 1 / rank

# Final MAP@3
map3_score /= len(train)

print(f"TF-IDF Pipeline MAP@3: {map3_score:.4f}")

**Milestone 2**

Q1, Introduction to Hugging Face transformers and datasets
Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [ ]:
#Installing the dataset library

!pip install datasets -q

In [ ]:
# Importing the library
from datasets import load_dataset

#Loading the training dataset
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train_ds = dataset["train"]

In [ ]:
#Create the combined_text column using .map()

def combine_prompt_option(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train_ds = train_ds.map(combine_prompt_option)

In [ ]:
#Find the character length of row 51

length = len(train_ds[51]["combined_text"])

print("Character Length:", length)

Q2, Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer? 

In [ ]:
# Install Transformers
!pip install transformers -q

In [ ]:
# Importing Autotokenizer
from transformers import AutoTokenizer

#Loading the BERT Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
print("Vocabulary Size:", tokenizer.vocab_size)

Q3, Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [ ]:
# Get the token ID of the [SEP] token
sep_token_id = tokenizer.sep_token_id

print("SEP Token ID:", sep_token_id)

Q4, Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [ ]:
#Converting the column to a Python list before tokenizing
prompts = train_ds["prompt"]

# Tokenize the entire prompt column
encoded = tokenizer(
    list(prompts),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Display the shape of the input_ids tensor
print("Input IDs Shape:", encoded["input_ids"].shape)

Q5, BERT/RoBERTa Architecture & Attention Mechanisms
A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head? 

In [ ]:
hidden_size = 768
num_attention_heads = 12

head_dimension = hidden_size // num_attention_heads

print("Dimension of each attention head:", head_dimension)

Q6, Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [ ]:
from transformers import AutoModel

# Load the BERT model
model = AutoModel.from_pretrained("bert-base-uncased")

#get the prompt from row 0
prompt = train_ds[0]["prompt"]

In [ ]:
#tokenize
inputs = tokenizer(prompt, return_tensors="pt")

#Passing through bERT
outputs = model(**inputs)

#Chekcing the shape
print(outputs.last_hidden_state.shape)

Q7, Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places). 

In [ ]:
# Extract the embedding of the [CLS] token
cls_embedding = outputs.last_hidden_state[0, 0]

# Display the first five values
print("First 5 values:")
print(cls_embedding[:5])

# Calculate the sum of the first five values
answer = cls_embedding[:5].sum().item()

print("Answer:", round(answer, 4))

Q8, Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places). 

In [ ]:
# Load the model with attention output
from transformers import AutoTokenizer, AutoModel

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load model and enable attention outputs
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

In [ ]:
# Tokenize the sentence
sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

In [ ]:
#pass through BERT
outputs = model(**inputs)

#Display the tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for i, token in enumerate(tokens):
    print(i, token)

In [ ]:
#Finding the index of 'Fusion'
fusion_index = tokens.index("fusion")

print("Fusion Token Index:", fusion_index)

In [ ]:
#Extract the Attention Matrix
# Last transformer layer
last_layer_attention = outputs.attentions[-1]

print(last_layer_attention.shape)

In [ ]:
# First Attention Head
head0 = last_layer_attention[0, 0]

# Attention from[CLS] to Fusion
attention_weight = head0[0, fusion_index].item()

print("Attention Weight:", round(attention_weight, 4))

Q9, Context-Aware Embeddings 
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [ ]:
# Installing the sentence transformer
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer, util

#load the model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# Extract the prompt and the option B
prompt = train_ds[0]["prompt"]
option_b = train_ds[0]["B"]

In [ ]:
# generating the sentence embedding
prompt_embedding = model.encode(prompt, convert_to_tensor=True)

option_b_embedding = model.encode(option_b, convert_to_tensor=True)

In [ ]:
# Computing the cosine similarity
similarity = util.cos_sim(
    prompt_embedding,
    option_b_embedding
)

print("Cosine Similarity:", round(similarity.item(), 4))

Q10, Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [ ]:
# Load the sentence Transformer
from sentence_transformers import SentenceTransformer, util

# Load the MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# MiniLM Ranking Pipeline


minilm_predictions = []

for row in train_ds:

    # Encode the question prompt
    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    similarities = []

    # Compare with each answer option
    for option in ["A", "B", "C", "D", "E"]:

        option_embedding = model.encode(
            row[option],
            convert_to_tensor=True
        )

        similarity = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()

        similarities.append((option, similarity))

    # Rank by cosine similarity
    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [x[0] for x in similarities[:3]]

    minilm_predictions.append(top3)

In [ ]:
# Calculate MAP@# Score
minilm_score = mapk(
    train_ds["answer"],
    minilm_predictions
)

print("MiniLM MAP@3:", round(minilm_score, 4))

In [ ]:
# Improvement Count
improved_count = 0

for i in range(len(train_ds)):

    actual = train_ds[i]["answer"]

    tfidf_correct = actual in predictions[i]

    minilm_correct = actual in minilm_predictions[i]

    if (not tfidf_correct) and minilm_correct:
        improved_count += 1

print(improved_count)

Q11,Zero-shot classification concepts 
Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [ ]:
#import pipeline
from transformers import pipeline

# Initialize zero shot pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
#Extract Prompt and  Candidiate lables
sequence = train_ds[1]["prompt"]

candidate_labels = [
    train_ds[1]["A"],
    train_ds[1]["B"],
    train_ds[1]["C"]
]

In [ ]:
#Perform zero shot Classification
result = classifier(
    sequence,
    candidate_labels
)

print(result)

top_score = result["scores"][0]

print("Top Probability:", round(top_score, 4))

Q12, Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [ ]:
# Run the classifer with multilabel = true

result_multilabel = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

print(result_multilabel)

In [ ]:
# Calculating the sums

# Sum of probabilities from Softmax (Q11)
softmax_sum = sum(result["scores"])

# Sum of probabilities from Independent Sigmoids (Q12)
sigmoid_sum = sum(result_multilabel["scores"])

print("Softmax Sum :", softmax_sum)
print("Sigmoid Sum :", sigmoid_sum)

In [ ]:
# Absolute Difference
difference = abs(softmax_sum - sigmoid_sum)

print("Absolute Difference:", round(difference, 4))

Q13, Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [ ]:
#Load FLAN-T5 Small

#generator = pipeline("text2text-generation",model="google/flan-t5-small")

In [ ]:
#import transformers
#print(transformers.__version__)

In [ ]:
# Constructing the prompt
'''prompt = (
    f"Question: {train_ds[0]['prompt']}. "
    f"Is the correct answer A: {train_ds[0]['A']} "
    f"or B: {train_ds[0]['B']}? "
    f"Answer with just the letter A or B."
)'''

In [ ]:
# Generate Response
'''result = generator(
    prompt,
    max_new_tokens=5
)

print(result)'''

In [ ]:
# Generate only the text
#answer = result[0]["generated_text"]

#print("Model Output:", answer)